In [1]:
import pyspark
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder \
        .master("local[*]") \
        .appName('test') \
        .getOrCreate()


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


25/03/06 19:27:10 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
df = spark.read.parquet('yellow/2024/01/')

In [4]:
from pyspark.sql.functions import col, count


In [5]:
pickup_counts = df.groupBy("PULocationID").agg(count("*").alias("count"))

In [6]:
least_frequent_zone = pickup_counts.orderBy(col("count").asc()).limit(1)

In [7]:
least_frequent_zone.show()


+------------+-----+
|PULocationID|count|
+------------+-----+
|         105|    1|
+------------+-----+



In [8]:
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-10.parquet


--2025-03-06 19:34:34--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-10.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 108.138.245.96, 108.138.245.225, 108.138.245.16, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|108.138.245.96|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 64346071 (61M) [binary/octet-stream]
Saving to: ‘yellow_tripdata_2024-10.parquet’

yellow_tripdata_202 100%[===================>]  61.36M   170MB/s    in 0.4s    

2025-03-06 19:34:35 (170 MB/s) - ‘yellow_tripdata_2024-10.parquet’ saved [64346071/64346071]



In [10]:
df = spark.read \
    .option("header", "true") \
    .parquet('yellow_tripdata_2024-10.parquet')

In [11]:
df = df.repartition(4)

In [12]:
df.write.parquet('yellow/2024/01/')

In [13]:
df = spark.read.parquet('yellow/2024/01/')

In [14]:
from pyspark.sql import functions as F

In [16]:
df_filtered = df.filter(F.to_date(F.col('tpep_pickup_datetime')) == '2024-10-15')

In [17]:
df_filtered = df_filtered.count()

In [18]:
print(f"Number of passengers picked up on Oct 15 2025 are {df_filtered}")

Number of passengers picked up on Oct 15 2025 are 128893


In [24]:
df_Hours = df.withColumn(
    'trip_duration', 
    ((F.unix_timestamp('tpep_dropoff_datetime') - F.unix_timestamp('tpep_pickup_datetime'))/3600)
)
longest_trip = df_Hours.orderBy(F.col('trip_duration').desc()).first()
print(f"Longest trip details: {longest_trip['trip_duration']}")

Longest trip details: 162.61777777777777


In [26]:
zone_lookup_df = spark.read.option("header", "true").csv("taxi_zone_lookup.csv")


In [27]:
pickup_counts = df.groupBy("PULocationID").agg(count("*").alias("count"))

In [31]:
 result = pickup_counts.join(zone_lookup_df, pickup_counts["PULocationID"] == zone_lookup_df["LocationID"], "left") \
    ...:                       .select(col("Zone"), col("count"))

In [32]:
least_frequent_zone = result.orderBy(col("count").asc()).limit(1)

In [33]:
least_frequent_zone.show()

+--------------------+-----+
|                Zone|count|
+--------------------+-----+
|Governor's Island...|    1|
+--------------------+-----+

